# 1) Imports & chargement

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import KNNImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.exceptions import NotFittedError

In [2]:
df_ventes = pd.read_csv('data/ech_annonces_ventes_68.csv', sep=';', index_col='idannonce')

# 2) Nettoyage et conversion de float à int

In [3]:
def clean_df(df: pd.DataFrame):
    # On supprime les colonnes ayant plus de 80% de valeurs manquantes
    valeurs_manquantes = df.isna().sum()/df.shape[0]
    valeurs_manquantes_list = list(valeurs_manquantes[valeurs_manquantes > 0.8].index)
    df_filtered = df.drop(valeurs_manquantes_list, axis = 1)
    # On supprime les colonnes "n6" pour ne laisser que les "n7" qui paraissent plus judicieuses (basées sur un plus grand nombre de bien)
    column_n6 = [ column for column in df_filtered.columns if "n6" in column]
    df_filtered = df_filtered.drop(column_n6, axis = 1)
    if 'typedebien' in df_filtered.columns:
        df_filtered = df_filtered[df_filtered['typedebien'] != 'l']
        df_filtered['typedebien'] = df_filtered['typedebien'].replace({'an': 'a', 'mn': 'm'})

    # On transform en Int des colonnes déclarées en float mais n'ayant que des int
    valeurs_manq_resid_quanti = [col for  col in df_filtered.select_dtypes(exclude='object').columns if df_filtered[col].isna().sum() > 0]    
    col_float = df_filtered[valeurs_manq_resid_quanti].select_dtypes(include='float64').columns
    for col in col_float:
        array_col = np.array(df_filtered[df_filtered[col].notna()][col]) 
        array_col_round = np.round(array_col)
        array_real_float = array_col[array_col != array_col_round]
        if len(array_real_float) == 0:
            df_filtered[col] = df_filtered[col].astype('Int64')
    # on transforme les colonnes quali n'ayant en fait que deux modalités
    if 'cave' in df_filtered.columns:
        df_filtered["cave"] = df_filtered["cave"].astype('Int64')
    if 'ascenseur' in df_filtered.columns:
        df_filtered["ascenseur"] = df_filtered["ascenseur"].astype('Int64')
    if 'logement_neuf' in df_filtered.columns:
        df_filtered["logement_neuf"] = df_filtered["logement_neuf"].replace({'n': False, 'o': True}).astype('Int64')

    return df_filtered

In [4]:
df_ventes = clean_df(df_ventes)
target = df_ventes['prix_bien']

# 3) Split train/test

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df_ventes.drop(columns=['prix_bien']),
    target,
    test_size=0.2,
    random_state=42
)

# 4) Transformer pour le remplissage (quali et quanti) du dataframe

In [6]:
class KNNImputerCustom(BaseEstimator, TransformerMixin):
    def __init__(self, type_col="typedebien", lat_col="mapCoordonneesLatitude", lon_col="mapCoordonneesLongitude", pieces_col="nb_pieces", k=10):
        self.type_col = type_col
        self.lat_col = lat_col
        self.lon_col = lon_col
        self.pieces_col = pieces_col
        self.k = k
        self.fitted = False
 
    def fit(self, X, y=None):
        self.df_train_ = X.copy()
        self.numeric_cols_ = X.select_dtypes(exclude="object").columns.tolist()
        self.default_mean_ = {col: X[col].mean() for col in self.numeric_cols_}
        self.imputer_ = KNNImputer()
        self.quali_cols_ = X.select_dtypes(include="object").columns.tolist()
        self.default_mode_ = {col: X[col].mode()[0] for col in self.quali_cols_}
        self.fitted = True
        
        return self
 
    def transform(self, X):
        if not(self.fitted):
            raise NotFittedError("This KNNImputerCustomed instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.")
        X = X.copy()
        # First we fill the numeric columns
        for col in self.numeric_cols_:
            if X[col].isna().sum() == 0:
                continue
            cols = [col, self.pieces_col, self.lat_col, self.lon_col]
            df_extract = X[[self.type_col] + cols]
            df_extract_train = self.df_train_ [[self.type_col] + cols]
            for typ in ["a", "m"]:
                subset = df_extract[df_extract[self.type_col] == typ][cols]
                if subset.shape[0] == 0:
                    continue
                apply_default = True
                subset_train = df_extract_train[df_extract_train[self.type_col] == typ][cols]
                if subset_train.shape[0] != 0:
                    self.imputer_.fit(subset_train)
                    imputed = self.imputer_.transform(subset)
                    if imputed.shape[1] == len(cols):
                        imputed_df = pd.DataFrame(imputed, columns=cols, index=subset.index)
                        apply_default = False
                if apply_default:
                    if X[col].dtype in ["int64", "Int64"]:
                        X.loc[subset.index, col] = self.default_mean_[col].astype("int64")
                    else:
                        X.loc[subset.index, col] = self.default_mean_[col]
                else:
                    if X[col].dtype in ["int64", "Int64"]:
                        X.loc[imputed_df.index, col] = imputed_df[col].round().astype("int64")
                    else:
                        X.loc[imputed_df.index, col] = imputed_df[col]

        #Secondly we fill the qualitative columns
        index_na = X[X[self.quali_cols_].isna().any(axis=1)].index
        for idx in index_na:
            lat = X.loc[idx, self.lat_col]
            lon = X.loc[idx, self.lon_col]
            nbp = X.loc[idx, self.pieces_col]
            typ = X.loc[idx, self.type_col]
            neighbours = self.df_train_[
                (self.df_train_[self.pieces_col] == nbp) &
                (self.df_train_[self.type_col] == typ)
            ]
            distances = (lat - neighbours[self.lat_col])**2 + (lon - neighbours[self.lon_col])**2
            for col in self.quali_cols_:
                if pd.isna(X.loc[idx, col]):
                    valid = neighbours[neighbours[col].notna()]
                    if valid.shape[0] > 0:
                        idx_neigh = distances[valid.index].sort_values().iloc[:self.k].index
                        X.loc[idx, col] = neighbours.loc[idx_neigh][col].mode()[0]
                    else:
                        X.loc[idx, col] = self.default_mode_[col]
        
        return X

In [7]:
knn_imputer_custom = KNNImputerCustom()
X_train = knn_imputer_custom.fit_transform(X_train)
X_test = knn_imputer_custom.transform(X_test)

# 5) Fonction de nettoyage des outiliers (IQR)

In [8]:
# Suppression de certains champs dont nous ne connaissons pas la définition
X_train.drop(columns=['duree_int', 'loyer_m2_median_n7', 'nb_log_n7', 'taux_rendement_n7'], axis=1, inplace=True)
X_test.drop(columns=['duree_int', 'loyer_m2_median_n7', 'nb_log_n7', 'taux_rendement_n7'], axis=1, inplace=True)

In [9]:
X_train_clean = X_train

numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()

index_to_drop = []

for col in numeric_cols:
    q1 = max(X_train[col].quantile(0.1), 0)
    q3 = max(X_train[col].quantile(0.9), 0)
    iqr = q3 - q1
    lower = max(q1 - 2 * iqr, 0)
    upper = max(q3 + 2 * iqr, 0)
    temp_index_to_drop = X_train[(X_train[col] < lower) | (X_train[col] > upper)].index.to_list()
    index_to_drop.extend(temp_index_to_drop)
    print("Suppression des lignes où ", col , " < ", lower, " ou ", col, " > ", upper, " (", len(temp_index_to_drop), " rows)")

X_train_clean = X_train_clean.drop(index_to_drop)

nb_rows_avant = X_train.shape[0]
nb_rows_apres = X_train_clean.shape[0]
nb_rows_suppr = nb_rows_avant - nb_rows_apres

print("\ndataframe initial : ", nb_rows_avant, " rows.")
print("dataframe nettoyé avec iqr : ", nb_rows_apres, " rows.")
print("Proportion conservé : ", 100 * np.round(nb_rows_apres/nb_rows_avant, 5), "%")
print("Nombre d'observations supprimées : ", nb_rows_suppr)

X_train = X_train_clean
y_train = y_train.loc[X_train.index]

Suppression des lignes où  etage  <  0  ou  etage  >  6.0  ( 272  rows)
Suppression des lignes où  surface  <  0  ou  surface  >  421.0  ( 105  rows)
Suppression des lignes où  surface_terrain  <  0  ou  surface_terrain  >  2850.0  ( 351  rows)
Suppression des lignes où  nb_pieces  <  0  ou  nb_pieces  >  17.0  ( 47  rows)
Suppression des lignes où  mensualiteFinance  <  0.0  ou  mensualiteFinance  >  0.0  ( 331  rows)
Suppression des lignes où  balcon  <  0  ou  balcon  >  3.0  ( 3  rows)
Suppression des lignes où  eau  <  0  ou  eau  >  3.0  ( 16  rows)
Suppression des lignes où  bain  <  0  ou  bain  >  3.0  ( 89  rows)
Suppression des lignes où  dpeC  <  0  ou  dpeC  >  745.8  ( 13  rows)
Suppression des lignes où  mapCoordonneesLatitude  <  46.595870000000005  ou  mapCoordonneesLatitude  >  49.07131999999999  ( 0  rows)
Suppression des lignes où  mapCoordonneesLongitude  <  6.43378  ou  mapCoordonneesLongitude  >  8.263480000000001  ( 0  rows)
Suppression des lignes où  nb_etages 

In [10]:
X_test_clean = X_test

numeric_cols = X_test.select_dtypes(include=['number']).columns.tolist()

index_to_drop = []

for col in numeric_cols:
    q1 = max(X_test[col].quantile(0.1), 0)
    q3 = max(X_test[col].quantile(0.9), 0)
    iqr = q3 - q1
    lower = max(q1 - 2 * iqr, 0)
    upper = max(q3 + 2 * iqr, 0)
    temp_index_to_drop = X_test[(X_test[col] < lower) | (X_test[col] > upper)].index.to_list()
    index_to_drop.extend(temp_index_to_drop)
    print("Suppression des lignes où ", col , " < ", lower, " ou ", col, " > ", upper, " (", len(temp_index_to_drop), " rows)")

X_test_clean = X_test_clean.drop(index_to_drop)

nb_rows_avant = X_test.shape[0]
nb_rows_apres = X_test_clean.shape[0]
nb_rows_suppr = nb_rows_avant - nb_rows_apres

print("\ndataframe initial : ", nb_rows_avant, " rows.")
print("dataframe nettoyé avec iqr : ", nb_rows_apres, " rows.")
print("Proportion conservé : ", 100 * np.round(nb_rows_apres/nb_rows_avant, 5), "%")
print("Nombre d'observations supprimées : ", nb_rows_suppr)

X_test = X_test_clean
y_test = y_test.loc[X_test.index]

Suppression des lignes où  etage  <  0  ou  etage  >  6.0  ( 85  rows)
Suppression des lignes où  surface  <  0  ou  surface  >  430.0  ( 26  rows)
Suppression des lignes où  surface_terrain  <  0  ou  surface_terrain  >  2857.5868  ( 85  rows)
Suppression des lignes où  nb_pieces  <  0  ou  nb_pieces  >  17.0  ( 12  rows)


Suppression des lignes où  mensualiteFinance  <  0.0  ou  mensualiteFinance  >  0.0  ( 88  rows)
Suppression des lignes où  balcon  <  0  ou  balcon  >  3.0  ( 1  rows)
Suppression des lignes où  eau  <  0  ou  eau  >  3.0  ( 4  rows)
Suppression des lignes où  bain  <  0  ou  bain  >  3.0  ( 22  rows)
Suppression des lignes où  dpeC  <  0  ou  dpeC  >  748.9800000000002  ( 3  rows)
Suppression des lignes où  mapCoordonneesLatitude  <  46.59563  ou  mapCoordonneesLatitude  >  49.07168  ( 0  rows)
Suppression des lignes où  mapCoordonneesLongitude  <  6.445864  ou  mapCoordonneesLongitude  >  8.251978999999999  ( 0  rows)
Suppression des lignes où  nb_etages  <  0  ou  nb_etages  >  10.0  ( 95  rows)
Suppression des lignes où  places_parking  <  0  ou  places_parking  >  7.0  ( 26  rows)
Suppression des lignes où  cave  <  0  ou  cave  >  3.0  ( 0  rows)
Suppression des lignes où  annee_construction  <  1742.0  ou  annee_construction  >  2192.0  ( 23  rows)
Suppression des lignes où  nb

# 6) Normalisation des données GPS (fit sur train)

In [11]:
scaler_lat = StandardScaler()
scaler_lon = StandardScaler()

X_train['Latitude_scaled']  = scaler_lat.fit_transform(X_train[['mapCoordonneesLatitude']])
X_train['Longitude_scaled'] = scaler_lon.fit_transform(X_train[['mapCoordonneesLongitude']])

X_test['Latitude_scaled']  = scaler_lat.transform(X_test[['mapCoordonneesLatitude']])
X_test['Longitude_scaled'] = scaler_lon.transform(X_test[['mapCoordonneesLongitude']])

# 7) Target Encoding (fit sur train)

In [12]:
cols_te = ['INSEE_COM', 'typedebien_lite', 'nb_pieces']

mean_price_by_combo = (
    X_train.assign(prix_bien=y_train)
    .groupby(cols_te, as_index=False)['prix_bien']
    .mean()
    .rename(columns={'prix_bien': 'prix_bien_target_encoding'})
)

X_train = X_train.merge(mean_price_by_combo, on=cols_te, how='left')
X_test  = X_test.merge(mean_price_by_combo, on=cols_te, how='left')

X_test[['prix_bien_target_encoding']] = X_test[['prix_bien_target_encoding']].fillna(y_train.mean())

# 8) Extraction année + OHE (fit sur train)

In [13]:
X_train['date'] = pd.to_datetime(X_train['date'])
X_test['date'] = pd.to_datetime(X_test['date'])

X_train['annee'] = X_train['date'].dt.year
X_test['annee'] = X_test['date'].dt.year

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
annee_train = ohe.fit_transform(X_train[['annee']])
annee_test  = ohe.transform(X_test[['annee']])

annee_cols = ohe.get_feature_names_out(['annee'])

X_train = pd.concat([X_train, pd.DataFrame(annee_train, columns=annee_cols, index=X_train.index)], axis=1)
X_test  = pd.concat([X_test,  pd.DataFrame(annee_test,  columns=annee_cols, index=X_test.index)], axis=1)

# 9) Parsing exposition / chauffage / DPE / GES

In [14]:
def parse_exposition(df):
    df = df.copy()
    df['exposition_clean'] = (
        df['exposition'].astype(str)
        .str.lower()
        .str.replace(r'[,/\\\-]', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )
    df['expo_nord']  = df['exposition_clean'].str.contains(r'\bnord\b',  na=False).astype(int)
    df['expo_sud']   = df['exposition_clean'].str.contains(r'\bsud\b',   na=False).astype(int)
    df['expo_est']   = df['exposition_clean'].str.contains(r'\best\b',   na=False).astype(int)
    df['expo_ouest'] = df['exposition_clean'].str.contains(r'\bouest\b', na=False).astype(int)
    df['expo_inconnue'] = df['exposition_clean'].str.contains(r'0|nan', na=False).astype(int)
    return df

In [15]:
def parse_chauffage_systeme(df):
    df = df.copy()
    df['chauffage_systeme_clean'] = (
        df['chauffage_systeme'].astype(str)
        .str.lower()
        .str.replace(r'[,/\\\-]', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    df['chauf_radiateur']  = df['chauffage_systeme_clean'].str.contains(r'\bradiateur\b', na=False).astype(int)
    df['chauf_sol']        = df['chauffage_systeme_clean'].str.contains(r'\bsol\b', na=False).astype(int)
    df['chauf_convecteur'] = df['chauffage_systeme_clean'].str.contains(r'\bconvecteur\b', na=False).astype(int)
    df['chauf_poele_bois'] = df['chauffage_systeme_clean'].str.contains(r'poêle|poele', na=False).astype(int)
    df['chauf_pac']        = df['chauffage_systeme_clean'].str.contains(r'pompe à chaleur|pac', na=False).astype(int)
    df['chauf_clim_rev']   = df['chauffage_systeme_clean'].str.contains(r'climatisation', na=False).astype(int)
    df['chauf_cheminee']   = df['chauffage_systeme_clean'].str.contains(r'cheminée|cheminee', na=False).astype(int)
    df['chauf_inconnu']    = df['chauffage_systeme_clean'].str.contains(r'nan', na=False).astype(int)
    return df

In [16]:
def parse_chauffage_energie(df):
    df = df.copy()
    df['chauffage_energie_clean'] = (
        df['chauffage_energie'].astype(str)
        .str.lower()
        .str.replace(r'[,/\\\-]', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    df['energie_gaz']   = df['chauffage_energie_clean'].str.contains(r'\bgaz\b', na=False).astype(int)
    df['energie_elec']  = df['chauffage_energie_clean'].str.contains(r'électrique|electrique', na=False).astype(int)
    df['energie_fioul'] = df['chauffage_energie_clean'].str.contains(r'\bfioul\b', na=False).astype(int)
    df['energie_bois']  = df['chauffage_energie_clean'].str.contains(r'\bbois\b', na=False).astype(int)
    df['energie_inconnue'] = df['chauffage_energie_clean'].str.contains(r'nan', na=False).astype(int)
    return df

In [17]:
def parse_chauffage_mode(df):
    df = df.copy()
    df['chauffage_mode_clean'] = (
        df['chauffage_mode'].astype(str)
        .str.lower()
        .str.replace(r'[,/\\\-]', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    df['chauffage_mode_individuel'] = df['chauffage_mode_clean'].str.contains(r'\bindividuel\b', na=False).astype(int)
    df['chauffage_mode_collectif']  = df['chauffage_mode_clean'].str.contains(r'\bcollectif\b', na=False).astype(int)
    df['chauffage_mode_central']    = df['chauffage_mode_clean'].str.contains(r'\bcentral\b', na=False).astype(int)
    df['chauffage_mode_inconnu']    = df['chauffage_mode_clean'].str.contains(r'nan', na=False).astype(int)
    return df

In [18]:
def parse_dpe(df):
    df = df.copy()
    df['dpe_A'] = df['dpeL'].str.contains(r'A', na=False).astype(int)
    df['dpe_B'] = df['dpeL'].str.contains(r'B', na=False).astype(int)
    df['dpe_C'] = df['dpeL'].str.contains(r'C', na=False).astype(int)
    df['dpe_D'] = df['dpeL'].str.contains(r'D', na=False).astype(int)
    df['dpe_E'] = df['dpeL'].str.contains(r'E', na=False).astype(int)
    df['dpe_F'] = df['dpeL'].str.contains(r'F', na=False).astype(int)
    df['dpe_G'] = df['dpeL'].str.contains(r'G', na=False).astype(int)
    df['dpe_inconnu'] = (~df['dpeL'].isin(list("ABCDEFG"))).astype(int)
    return df

In [19]:
def parse_ges(df):
    df = df.copy()
    df['ges_A'] = df['ges_class'].str.contains(r'A', na=False).astype(int)
    df['ges_B'] = df['ges_class'].str.contains(r'B', na=False).astype(int)
    df['ges_C'] = df['ges_class'].str.contains(r'C', na=False).astype(int)
    df['ges_D'] = df['ges_class'].str.contains(r'D', na=False).astype(int)
    df['ges_E'] = df['ges_class'].str.contains(r'E', na=False).astype(int)
    df['ges_F'] = df['ges_class'].str.contains(r'F', na=False).astype(int)
    df['ges_G'] = df['ges_class'].str.contains(r'G', na=False).astype(int)
    df['ges_inconnu'] = (~df['ges_class'].isin(list("ABCDEFG"))).astype(int)
    return df

In [20]:
X_train = parse_exposition(X_train)
X_test  = parse_exposition(X_test)

X_train = parse_chauffage_systeme(X_train)
X_test  = parse_chauffage_systeme(X_test)

X_train = parse_chauffage_energie(X_train)
X_test  = parse_chauffage_energie(X_test)

X_train = parse_dpe(X_train)
X_test  = parse_dpe(X_test)

X_train = parse_ges(X_train)
X_test  = parse_ges(X_test)

X_train = parse_chauffage_mode(X_train)
X_test  = parse_chauffage_mode(X_test)

# 10) OHE sur le reste des variables catégorielles

In [21]:
# --- Liste des colonnes à exclure (déjà parsées ou inutiles)
exclude_cols = [
    'mapCoordonneesLatitude', 'mapCoordonneesLongitude',
    'date', 'annee',
    'exposition', 'exposition_clean',
    'chauffage_systeme', 'chauffage_systeme_clean',
    'chauffage_energie', 'chauffage_energie_clean',
    'dpeL', 'ges_class',
    'chauffage_mode', 'chauffage_mode_clean'
]

# --- Sélection des colonnes catégorielles restantes
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns
cat_cols = [c for c in cat_cols if c not in exclude_cols]

print("Colonnes catégorielles encodées :", cat_cols)

# --- OneHotEncoder (fit sur train)
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

ohe_train = ohe.fit_transform(X_train[cat_cols])
ohe_test  = ohe.transform(X_test[cat_cols])

ohe_cols = ohe.get_feature_names_out(cat_cols)

# --- Ajout des colonnes encodées
X_train_ohe = pd.DataFrame(ohe_train, columns=ohe_cols, index=X_train.index)
X_test_ohe  = pd.DataFrame(ohe_test,  columns=ohe_cols, index=X_test.index)

X_train = pd.concat([X_train.drop(columns=cat_cols+exclude_cols), X_train_ohe], axis=1)
X_test  = pd.concat([X_test.drop(columns=cat_cols+exclude_cols),  X_test_ohe], axis=1)

Colonnes catégorielles encodées : ['type_annonceur', 'typedebien', 'typedetransaction', 'annonce_exclusive', 'categorie_annonceur', 'typedebien_lite', 'TYP_IRIS_x', 'TYP_IRIS_y']


In [22]:
pd.set_option('display.max_columns', 100)

In [23]:
X_train.head()

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,chauf_radiateur,chauf_sol,chauf_convecteur,chauf_poele_bois,chauf_pac,chauf_clim_rev,chauf_cheminee,chauf_inconnu,energie_gaz,energie_elec,energie_fioul,energie_bois,energie_inconnue,dpe_A,dpe_B,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
0,1,74,454.206,3,0,0,0,0,166.0,3,1,1,2005,1,1,12,1967.000,0,68135,0,681350000,6813500,68403,44,68,2814.86,-1.268439,1.437389,258272.800000,0.0,0.0,1.0,0.0,0.0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
1,0,122,1239.000,5,0,0,1,0,240.0,2,5,0,1974,2,0,7,70.534,0,68315,101,683150101,6831501,68401,44,68,3114.75,0.502046,-0.903823,306629.411765,0.0,0.0,0.0,1.0,0.0,1,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,0,167,560.000,5,0,0,0,0,400.0,2,3,0,1970,1,0,4,4.800,0,68041,0,680410000,6804100,68000,44,68,772.46,0.412261,1.455061,298247.958333,0.0,0.0,1.0,0.0,0.0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
3,0,129,1247.000,6,0,1,0,1,299.0,2,2,1,1984,2,0,14,988.800,0,68309,0,683090000,6830900,68115,44,68,3093.02,-0.839371,0.887967,451673.411765,0.0,0.0,0.0,1.0,0.0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
4,3,82,550.000,4,0,0,0,1,186.2,4,1,1,1799,1,1,80,2400.000,0,68224,201,682240201,6822401,68701,44,68,1195.12,-0.348123,-0.066819,161106.385017,0.0,0.0,1.0,0.0,0.0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


In [24]:
X_test.head()

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,chauf_radiateur,chauf_sol,chauf_convecteur,chauf_poele_bois,chauf_pac,chauf_clim_rev,chauf_cheminee,chauf_inconnu,energie_gaz,energie_elec,energie_fioul,energie_bois,energie_inconnue,dpe_A,dpe_B,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
0,0,96,110.074,5,0,0,1,0,153.0,2,1,1,1970,1,0,30,3500.0,0,68300,102,683000102,6830001,68701,44,68,1202.08,-0.171263,0.200939,465000.000000,0.0,0.0,0.0,1.0,0.0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1,0,58,622.600,3,0,1,1,0,197.0,2,1,1,1914,1,0,15,240.0,0,68334,104,683340104,6833401,68402,44,68,1836.21,0.043050,-1.846239,142705.915254,0.0,1.0,0.0,0.0,0.0,1,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
2,1,61,104.000,3,0,0,0,1,250.0,2,2,1,1954,1,1,14,0.0,0,68334,102,683340102,6833401,68402,44,68,1938.52,-0.015535,-1.827338,142705.915254,0.0,1.0,0.0,0.0,0.0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,0,74,569.600,4,0,0,0,1,216.0,4,1,1,1950,1,0,16,948.0,0,68224,1203,682241203,6822401,68701,44,68,1027.03,-0.315539,-0.102853,161106.385017,0.0,0.0,0.0,0.0,1.0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
4,4,166,344.000,7,0,0,1,1,156.0,4,1,1,1881,2,1,14,1389.0,0,68237,0,682370000,6823700,68000,44,68,2349.40,1.638888,-0.760455,270071.340026,0.0,0.0,1.0,0.0,0.0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0


# 11) Standardisation des données

In [25]:
df_scaler = StandardScaler()

train_columns = X_train.columns
train_index = X_train.index
test_index = X_test.index

X_train = pd.DataFrame(data=df_scaler.fit_transform(X_train), index=train_index, columns=train_columns)
X_test = pd.DataFrame(data=df_scaler.transform(X_test), index=test_index, columns=train_columns)

In [26]:
display(X_train.head())
X_train.describe()

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,chauf_radiateur,chauf_sol,chauf_convecteur,chauf_poele_bois,chauf_pac,chauf_clim_rev,chauf_cheminee,chauf_inconnu,energie_gaz,energie_elec,energie_fioul,energie_bois,energie_inconnue,dpe_A,dpe_B,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
0,0.413830,-0.620560,-0.069990,-0.808839,0.0,-0.390623,-0.540204,-1.040163,-0.471184,0.437454,-0.789899,0.719961,0.886949,-0.757060,1.627155,-0.456053,1.415073,-0.412419,-0.543565,-0.503463,-0.543725,-0.543611,0.128846,0.0,0.0,0.194809,-1.268439,1.437389,-0.093318,-0.136641,-0.555018,1.698413,-0.721362,-0.414273,-0.185887,-0.413951,-0.266815,-0.316050,0.545645,0.334242,-0.313438,-0.073325,-0.058016,-0.055428,-0.031197,-0.020922,0.0,-1.374669,-0.435722,2.109546,-0.068477,0.0,-0.221383,-0.249017,-0.409429,1.925322,-0.410723,-0.237355,-0.132522,-0.692204,-0.394825,-0.377534,-0.461994,-0.503280,2.334953,-0.285478,-0.200903,-0.320681,0.390619,-0.397781,-0.044691,0.0,0.0,1.009035,-1.009035,-0.013947,0.110922,-0.110021,-0.456561,-0.825824,1.170225,0.581898,-0.038851,-0.196007,-0.366354,-0.300062,-0.11705,1.009035,-1.009035,-0.027015,-0.998688,1.000146,-1.000146,1.000146
1,-0.498014,0.352451,1.882144,0.318406,0.0,-0.390623,1.422603,-1.040163,0.380930,-0.302209,2.898925,-1.388965,0.067393,0.911962,-0.614570,-0.590009,-1.250280,-0.412419,1.136033,-0.195788,1.135986,1.136078,0.121434,0.0,0.0,0.505605,0.502046,-0.903823,0.289150,-0.136641,-0.555018,-0.588785,1.386266,-0.414273,5.379621,-0.413951,-0.266815,3.164055,-1.832693,0.334242,-0.313438,-0.073325,-0.058016,-0.055428,-0.031197,-0.020922,0.0,0.727448,-0.435722,-0.474036,-0.068477,0.0,-0.221383,-0.249017,-0.409429,1.925322,-0.410723,-0.237355,-0.132522,-0.692204,-0.394825,2.648769,-0.461994,-0.503280,-0.428274,-0.285478,-0.200903,-0.320681,0.390619,-0.397781,-0.044691,0.0,0.0,-0.991046,0.991046,-0.013947,0.110922,-0.110021,-0.456561,-0.825824,1.170225,0.581898,-0.038851,-0.196007,-0.366354,-0.300062,-0.11705,-0.991046,0.991046,-0.027015,1.001314,-0.999854,0.999854,-0.999854
2,-0.498014,1.264649,0.193167,0.318406,0.0,-0.390623,-0.540204,-1.040163,2.223338,-0.302209,1.054513,-1.388965,-0.038356,-0.757060,-0.614570,-0.670382,-1.342665,-0.412419,-1.420688,-0.503463,-1.420859,-1.420734,-1.364713,0.0,0.0,-1.921866,0.412261,1.455061,0.222858,-0.136641,-0.555018,1.698413,-0.721362,-0.414273,-0.185887,-0.413951,-0.266815,-0.316050,0.545645,0.334242,-0.313438,-0.073325,-0.058016,-0.055428,-0.031197,-0.020922,0.0,-1.374669,2.295041,-0.474036,-0.068477,0.0,-0.221383,-0.249017,-0.409429,-0.519394,-0.410723,4.213102,-0.132522,-0.692204,-0.394825,-0.377534,-0.461994,-0.503280,-0.428274,-0.285478,4.977518,-0.320681,0.390619,-0.397781,-0.044691,0.0,0.0,-0.991046,0.991046,-0.013947,0.110922,-0.110021,-0.456561,-0.825824,1.170225,0.581898,-0.038851,-0.196007,-0.366354,-0.300062,-0.11705,-0.991046,0.991046,-0.027015,-0.998688,1.000146,-1.000146,1.000146
3,-0.498014,0.494348,1.902044,0.882028,0.0,2.085409,-0.540204,0.552565,1.060318,-0.302209,0.132307,0.719961,0.331766,0.911962,-0.614570,-0.402471,0.0

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,chauf_radiateur,chauf_sol,chauf_convecteur,chauf_poele_bois,chauf_pac,chauf_clim_rev,chauf_cheminee,chauf_inconnu,energie_gaz,energie_elec,energie_fioul,energie_bois,energie_inconnue,dpe_A,dpe_B,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
count,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,20569.0,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,20569.0,20569.0,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,20569.0,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,20569.0,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,20569.000000,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,20569.0,20569.0,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04,2.056900e+04
mean,-8.290644e-18,-5.354374e-18,-2.487193e-17,3.108992e-18,0.0,-7.323402e-17,4.836209e-18,1.022513e-16,5.057293e-16,8.014289e-17,6.597971e-17,-6.425249e-17,4.881117e-16,-1.506134e-16,8.290644e-17,-7.599757e-18,1.727218e-16,-3.316258e-17,6.744767e-14,-1.381774e-18,-1.649208e-14,6.067974e-15,1.667801e-14,0.0,0.0,1.658129e-16,-5.527096e-18,5.527096e-18,1.671947e-16,3.661701e-17,1.519951e-17,1.105419e-17,2.901725e-17,-5.043475e-17,-7.461580e-17,-2.003572e-17,6.217983e-17,9.119708e-17,-9.257886e-17,-2.176294e-17,-7.599757e-18,4.836209e-18,-4.093505e-17,-8.290644e-18,7.599757e-18,-7.599757e-18,0.0,7.254314e-17,-3.108992e-18,-2.521738e-17,2.176294e-17,0.0,1.036331e-17,3.316258e-17,-5.250741e-17,-5.008931e-18,-5.388919e-17,0.000000,-2.694459e-17,1.658129e-17,3.005358e-17,6.027989e-17,4.283499e-17,2.556282e-17,5.423463e-17,5.458007e-17,-2.832637e-17,3.039903e-17,-8.290644e-18,-5.043475e-17,-7.599757e-18,0.0,0.0,-1.001786e-16,1.354139e-16,-2.763548e-18,-5.803451e-17,3.730790e-17,-3.592612e-17,9.534241e-17,-1.036331e-18,-1.243597e-16,-1.312685e-17,2.763548e-18,-3.039903e-17,-5.907084e-17,1.226324e-17,-1.001786e-16,1.354139e-16,-4.145322e-18,-2.487193e-17,-1.450863e-17,1.450863e-17,-1.450863e-17
std,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,0.0,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,0.0,0.0,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.00002

In [27]:
display(X_test.head())
X_test.describe()

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,chauf_radiateur,chauf_sol,chauf_convecteur,chauf_poele_bois,chauf_pac,chauf_clim_rev,chauf_cheminee,chauf_inconnu,energie_gaz,energie_elec,energie_fioul,energie_bois,energie_inconnue,dpe_A,dpe_B,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
0,-0.498014,-0.174597,-0.926000,0.318406,0.0,-0.390623,1.422603,-1.040163,-0.620880,-0.302209,-0.789899,0.719961,-0.038356,-0.757060,-0.614570,0.026187,3.569600,-0.412419,0.996066,-0.192742,0.996019,0.996112,1.233264,0.0,0.0,-1.476622,-0.171263,0.200939,1.541754,-0.136641,-0.555018,-0.588785,1.386266,-0.414273,-0.185887,-0.413951,-0.266815,-0.316050,0.545645,0.334242,-0.313438,-0.073325,-0.058016,-0.055428,-0.031197,-0.020922,0.0,0.727448,-0.435722,-0.474036,-0.068477,0.0,-0.221383,-0.249017,-0.409429,1.925322,-0.410723,-0.237355,-0.132522,-0.692204,-0.394825,-0.377534,-0.461994,1.986967,-0.428274,-0.285478,-0.200903,-0.320681,-2.560039,2.513947,-0.044691,0.0,0.0,1.009035,-1.009035,-0.013947,0.110922,-0.110021,-0.456561,-0.825824,1.170225,0.581898,-0.038851,-0.196007,-0.366354,-0.300062,-0.11705,1.009035,-1.009035,-0.027015,1.001314,-0.999854,0.999854,-0.999854
1,-0.498014,-0.944897,0.348881,-0.808839,0.0,2.085409,1.422603,-1.040163,-0.114218,-0.302209,-0.789899,0.719961,-1.518844,-0.757060,-0.614570,-0.375680,-1.012107,-0.412419,1.313323,-0.186650,1.313281,1.313369,0.125140,0.0,0.0,-0.819431,0.043050,-1.846239,-1.007374,-0.136641,1.801745,-0.588785,-0.721362,-0.414273,5.379621,-0.413951,-0.266815,3.164055,-1.832693,0.334242,-0.313438,-0.073325,-0.058016,-0.055428,-0.031197,-0.020922,0.0,0.727448,-0.435722,-0.474036,-0.068477,0.0,-0.221383,-0.249017,-0.409429,1.925322,-0.410723,-0.237355,-0.132522,-0.692204,-0.394825,2.648769,-0.461994,-0.503280,-0.428274,-0.285478,-0.200903,-0.320681,0.390619,-0.397781,-0.044691,0.0,0.0,1.009035,-1.009035,-0.013947,0.110922,-0.110021,-0.456561,-0.825824,1.170225,0.581898,-0.038851,-0.196007,-0.366354,-0.300062,-0.11705,1.009035,-1.009035,-0.027015,1.001314,-0.999854,0.999854,-0.999854
2,0.413830,-0.884084,-0.941109,-0.808839,0.0,-0.390623,-0.540204,0.552565,0.496080,-0.302209,0.132307,0.719961,-0.461353,-0.757060,1.627155,-0.402471,-1.349411,-0.412419,1.313323,-0.192742,1.313280,1.313369,0.125140,0.0,0.0,-0.713400,-0.015535,-1.827338,-1.007374,-0.136641,1.801745,-0.588785,-0.721362,-0.414273,-0.185887,-0.413951,-0.266815,-0.316050,0.545645,0.334242,-0.313438,-0.073325,-0.058016,-0.055428,-0.031197,-0.020922,0.0,0.727448,-0.435722,-0.474036,-0.068477,0.0,-0.221383,-0.249017,-0.409429,-0.519394,2.434732,-0.237355,-0.132522,-0.692204,-0.394825,-0.377534,-0.461994,-0.503280,2.334953,-0.285478,-0.200903,-0.320681,0.390619,-0.397781,-0.044691,0.0,0.0,1.009035,-1.009035,-0.013947,0.110922,-0.110021,-0.456561,-0.825824,1.170225,0.581898,-0.038851,-0.196007,-0.366354,-0.300062,-0.11705,1.009035,-1.009035,-0.027015,1.001314,-0.999854,0.999854,-0.999854
3,-0.498014,-0.620560,0.217047,-0.245216,0.0,-0.390623,-0.540204,0.552565,0.104568,1.177117,-0.789899,0.719961,-0.567102,-0.757060,-0.614570,-0.348

,etage,surface,surface_terrain,nb_pieces,mensualiteFinance,balcon,eau,bain,dpeC,nb_etages,places_parking,cave,annee_construction,nb_toilettes,ascenseur,nb_logements_copro,charges_copro,logement_neuf,INSEE_COM,IRIS,CODE_IRIS,GRD_QUART,UU2010,REG,DEP,prix_m2_vente,Latitude_scaled,Longitude_scaled,prix_bien_target_encoding,annee_2019,annee_2020,annee_2021,annee_2022,annee_2023,expo_nord,expo_sud,expo_est,expo_ouest,expo_inconnue,chauf_radiateur,chauf_sol,chauf_convecteur,chauf_poele_bois,chauf_pac,chauf_clim_rev,chauf_cheminee,chauf_inconnu,energie_gaz,energie_elec,energie_fioul,energie_bois,energie_inconnue,dpe_A,dpe_B,dpe_C,dpe_D,dpe_E,dpe_F,dpe_G,dpe_inconnu,ges_A,ges_B,ges_C,ges_D,ges_E,ges_F,ges_G,ges_inconnu,chauffage_mode_individuel,chauffage_mode_collectif,chauffage_mode_central,chauffage_mode_inconnu,type_annonceur_pr,typedebien_a,typedebien_m,typedetransaction_pi,typedetransaction_v,typedetransaction_vp,annonce_exclusive_0,annonce_exclusive_Non,annonce_exclusive_Oui,categorie_annonceur_a,categorie_annonceur_b,categorie_annonceur_ca,categorie_annonceur_cm,categorie_annonceur_m,categorie_annonceur_network,typedebien_lite_a,typedebien_lite_m,TYP_IRIS_x_D,TYP_IRIS_x_H,TYP_IRIS_x_Z,TYP_IRIS_y_H,TYP_IRIS_y_Z
count,5127.000000,5127.000000,5127.000000,5127.000000,5127.0,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.0,5127.0,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.0,5127.000000,5127.000000,5127.000000,5127.000000,5127.0,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.0,5127.0,5127.000000,5127.000000,5.127000e+03,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000,5127.000000
mean,-0.006967,0.007376,-0.011557,0.003340,0.0,-0.007169,0.038263,-0.002576,0.008218,-0.022184,0.013592,0.000944,-0.000353,-0.005398,-0.022548,-0.013757,-0.027258,-0.022292,-0.007197,-0.000066,-0.007197,-0.007198,-0.014263,0.0,0.0,0.014362,0.000846,0.010291,0.007118,0.010221,0.003949,-0.015982,0.013245,-0.006628,0.007337,0.001645,0.008037,-0.007205,0.001044,0.005331,0.001616,-0.025188,0.002703,0.015166,-0.006165,-0.011596,0.0,0.014032,-0.012819,-0.006401,0.000204,0.0,-0.019905,0.015506,0.006641,0.002737,0.002748,0.017850,0.011252,-0.019927,-0.015100,0.035654,-0.019374,-0.007853,-0.000882,-0.012821,0.016253,0.015375,0.010780,-0.011027,-0.009706,0.0,0.0,-0.014997,0.014997,-1.394651e-02,0.018360,-0.016718,-0.035812,-0.009859,0.037196,0.012066,-0.008683,0.001358,-0.023970,0.013823,-0.010632,-0.014997,0.014997,-0.012564,-0.016827,0.017505,-0.017505,0.017505
std,1.010415,0.993910,0.989532,0.987204,0.0,0.973898,1.049564,0.999375,1.009986,0.983577,0.997320,0.999781,1.004481,1.010531,0.988357,0.987924,0.986504,0.977155,0.993166,0.997138,0.993165,0.993166,0.997030,0.0,0.0,1.019221,1.009189,0.990834,0.955509,1.036105,1.002549,0.991061,1.004404,0.993426,1.018949,1.001742,1.013959,0.989757,0.999425,0.992973,1.002418,0.811069,1.023045,1.128183,0.895930,0.667804,0.0,0.995447,0.988024,0.994828,1.001577,0.0,0.956178,1.028772,1.006804,1.002016,1.002871,1.034822,1.040914,0.992371,0.983707,1.039191,0.983275,0.994223,0.999256,0.979170,1.038066,1.021265,0.988275,0.988298,0.885043,0.0,0.0,0.999850,0.999850,3.469785e-18,0.914515,0.921829,0.967890,0.998149,1.005264,0.993143,0.881425,1.003423,0.971062,1.020751,0.954191,0.999850,0.999850,0.731578,0.999934,0.999947,0.999

# 12) Sélection de variables

## 12.1) Laila

## 12.2) Hugues

## 12.3) Matthieu

# 13) Modélisation

In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Modèle
lr = LinearRegression()
lr.fit(X_train, y_train)

# Prédictions
y_pred = lr.predict(X_test)

# Scores
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R2   : {r2:.4f}")


MAE  : 31428.47
RMSE : 50699.14
R2   : 0.8985
